## Preamble

In [ ]:
# Load libraries
import pandas as pd
import os
import sqlite3
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [ ]:
# Directory
os.chdir('/Users/tasia/Desktop/Data Analysis/Assessment')

In [ ]:
# Load and set up GDP and Unemployment data
# Load data from database
conn = sqlite3.connect("wdi_cross_sectional_gdp_unemployment_2023.db")

df_gdp = pd.read_sql_query("SELECT * FROM gdp", conn)
df_unp = pd.read_sql_query("SELECT * FROM unemployment", conn)

conn.close()

In [ ]:
# Remove rows with missing country codes
df_gdp = df_gdp.dropna(subset = ["Country Code"])
df_unp = df_unp.dropna(subset = ["Country Code"])

df_gdp = df_gdp[df_gdp["Country Code"] != ""]
df_unp = df_unp[df_unp["Country Code"] != ""]

# Merge dataframes (inner join!)
df = pd.merge(left = df_gdp,
              right = df_unp,
              how = "inner",
              on = "Country Code")

In [ ]:
# Rename columns
df = df.rename(columns = {"2023 [YR2023]_x": "gdp",
                          "2023 [YR2023]_y": "unemployment"})

# Convert columns to numeric and replace invalid values with NaN
df["gdp"] = pd.to_numeric(df["gdp"], errors = "coerce")
df["unemployment"] = pd.to_numeric(df["unemployment"], errors = "coerce")

# Remove missing values
df = df.dropna(subset = ["gdp", "unemployment"])

# Eyeball
df.head()

In [ ]:
# IMF advanced economies 
# Source: IMF World Economic Outlook (October 2023)
advanced = ["AND", "AUS", "AUT", "BEL", "CAN", "HRV",
            "CYP", "CZE", "DNK", "EST", "FIN", "FRA",
            "DEU", "GRC", "HKG", "ISL", "IRL", "ISR",
            "ITA", "JPN", "KOR", "LVA", "LTU", "LUX",
            "MAC", "MLT", "NLD", "NZL", "NOR", "PRT",
            "PRI", "SMR", "SGP", "SVK", "SVN", "ESP",
            "SWE", "CHE", "TWN", "GBR", "USA"]

In [ ]:
# IMF emerging market and developing economies
developing = ["AFG", "ALB", "DZA", "AGO", "ATG", "ARG",
              "ARM", "ABW", "AZE", "BHS", "BHR", "BGD",
              "BRB", "BLR", "BLZ", "BEN", "BTN", "BOL",
              "BIH", "BWA", "BRA", "BRN", "BGR", "BFA",
              "BDI", "CPV", "KHM", "CMR", "CAF", "TCD",
              "CHL", "CHN", "COL", "COM", "COD", "COG",
              "CRI", "CIV", "DJI", "DMA", "DOM", "ECU",
              "EGY", "SLV", "GNQ", "ERI", "SWZ", "ETH",
              "FJI", "GAB", "GMB", "GEO", "GHA", "GRD",
              "GTM", "GIN", "GNB", "GUY", "HTI", "HND",
              "HUN", "IND", "IDN", "IRN", "IRQ", "JAM",
              "JOR", "KAZ", "KEN", "KIR", "XKX", "KWT",
              "KGZ", "LAO", "LBN", "LSO", "LBR", "LBY",
              "MDG", "MWI", "MYS", "MDV", "MLI", "MHL",
              "MRT", "MUS", "MEX", "FSM", "MDA", "MNG",
              "MNE", "MAR", "MOZ", "MMR", "NAM", "NRU",
              "NPL", "NIC", "NER", "NGA", "MKD", "OMN",
              "PAK", "PLW", "PAN", "PNG", "PRY", "PER",
              "PHL", "POL", "QAT", "ROU", "RUS", "RWA",
              "WSM", "STP", "SAU", "SEN", "SRB", "SYC",
              "SLE", "SLB", "SOM", "ZAF", "SSD", "LKA",
              "KNA", "LCA", "VCT", "SDN", "SUR", "SYR",
              "TJK", "TZA", "THA", "TLS", "TGO", "TON",
              "TTO", "TUN", "TUR", "TKM", "TUV", "UGA",
              "UKR", "ARE", "URY", "UZB", "VUT", "VEN",
              "VNM", "PSE", "YEM", "ZMB", "ZWE"]

In [ ]:
# Keep economies classified by the IMF
df = df[df["Country Code"].isin(advanced + developing)].copy()

# Create classification variable
df["classification"] = 0
df.loc[df["Country Code"].isin(advanced), "classification"] = 1

# Eyeball
df[["Country Name_x", "gdp", "unemployment", "classification"]].head()

In [ ]:
# Check number of economies in each class
df["classification"].value_counts()

## Prepare Data for Classification

In [ ]:
# Define explanatory and dependent variables
X = df[["gdp", "unemployment"]]
y = df["classification"]

In [ ]:
# Split dataset into a training and testing sample
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                   test_size = 0.25,
                                                   random_state = 42,
                                                   stratify = y)

In [ ]:
# Standardise explanatory variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Model 1: Logistic Regression

In [ ]:
# Estimate logistic regression model
model1 = LogisticRegression()
model1.fit(X_train_scaled, y_train)

In [ ]:
# Get predictions
y_pred1 = model1.predict(X_test_scaled)

In [ ]:
# Confusion matrix
confusion_matrix(y_test, y_pred1)

In [ ]:
# Calculate accuracy
accuracy_score(y_test, y_pred1)

In [ ]:
# Classification report
print(classification_report(y_test, y_pred1))

### Interpretation and Discussing the Output

Based on the Logistic Regression results in Model 1 above, the model has an accuracy of 93.18 percent, meaning that 41 out of 44 observations within the testing sample are correctly classified.

Looking at the confusion matrix, one can see that 32 emerging market and developing economies and 9 advanced economies are correctly classified. But in contrast, 2 emerging markets and developing economies are incorrectly classified as advanced economies, whilst 1 advanced economy is incorrectly classified.

The precision and recall values for advanced economies are 0.82 and 0.90, respectively, with an F1-score of 0.86.

Overall, Model 1 demonstrates relatively high predictive accuracy in classifying economies based on GDP per capita and unemployment rates.

### Model 2: Linear Discriminant Analysis

In [ ]:
# Estimate linear discriminant analysis model
model2 = LinearDiscriminantAnalysis()
model2.fit(X_train_scaled, y_train)

In [ ]:
# Get predictions
y_pred2 = model2.predict(X_test_scaled)

In [ ]:
# Prior probabilities of groups
print(model2.priors_)

# Group means
print(model2.means_)

# Coefficients of linear discriminants
print(model2.scalings_)

In [ ]:
# Confusion matrix
confusion_matrix(y_test, y_pred2)

In [ ]:
# Calculate accuracy
accuracy_score(y_test, y_pred2)

In [ ]:
# Classification report
print(classification_report(y_test, y_pred2))

### Interpretation and Discussing the Output

Based on the Linear Discriminant Analysis results in Model 2, the prior probabilities for emerging markets and developing economies and advanced economies are 0.783 and 0.217, respectively, reflecting the unequal distribution of observations across the two groups within the training sample.

Looking at the group means, advanced economies have, on average, higher standardized GDP per capita and lower standardized unemployment rates relative to emerging market and developing economies. Furthermore, the coefficients of the linear discriminant suggest that GDP per capita contributes considerably more towards distinguishing between the two groups than unemployment.

Overall, Model 2 achieves an accuracy of 93.18 percent, with the same confusion matrix and classification report results as Model 1.

### Model 3: Probit Regression

In [ ]:
# Estimate probit model
model3 = sm.Probit(y_train, sm.add_constant(X_train_scaled)).fit()
model3.summary()

In [ ]:
# Predict probabilities
prob3 = model3.predict(sm.add_constant(X_test_scaled))

# Get predictions
y_pred3 = (prob3 > 0.5).astype(int)

In [ ]:
# Confusion matrix
confusion_matrix(y_test, y_pred3)

In [ ]:
# Calculate accuracy
accuracy_score(y_test, y_pred3)

In [ ]:
# Classification report
print(classification_report(y_test, y_pred3))

### Interpretation and Discussing the Output

Based on the Probit Regression results in Model 3 above, GDP per capita is statistically significant at the 1% level with a positive coefficient of 2.4108. This indicates that higher GDP per capita is associated with a higher probability of being classified as an advanced economy, ceteris paribus. In contrast, the coefficient for unemployment (0.1336) is not statistically significant at conventional significance levels, with a p-value of 0.658.

Looking at the confusion matrix, 31 emerging market and developing economies and 9 advanced economies are correctly classified, whilst 4 observations are incorrectly classified.

Overall, Model 3 achieves an accuracy of 90.91 percent, which is marginally lower relative to Models 1 and 2.

### Model 4: Support Vector Machine

In [ ]:
# Estimate support vector machine model
model4 = SVC(kernel = "linear", C = 11)
model4.fit(X_train_scaled, y_train)

In [ ]:
# Get predictions
y_pred4 = model4.predict(X_test_scaled)

In [ ]:
# Confusion matrix
confusion_matrix(y_test, y_pred4)

In [ ]:
# Calculate accuracy
accuracy_score(y_test, y_pred4)

In [ ]:
# Classification report
print(classification_report(y_test, y_pred4))

In [ ]:
# Plot support vector machine result (Figure 1)
# Creating a grid
x = np.linspace(X_train_scaled[:, 0].min(), X_train_scaled[:, 0].max(), 500)
y = np.linspace(X_train_scaled[:, 1].min(), X_train_scaled[:, 1].max(), 500)
xx, yy = np.meshgrid(x, y)

# Predict classifications across the grid
Z = model4.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot classification regions
plt.contourf(xx, yy, Z, levels = [-0.5, 0.5, 1.5], cmap = "RdYlBu", alpha = 0.3)

# Plot country observations
points = plt.scatter(X_train_scaled[:, 0],
                     X_train_scaled[:, 1],
                     c = y_train,
                     cmap = "RdYlBu",
                     edgecolors = "black")

# Adding labels, title and legend
plt.xlabel("Standardised GDP per capita")
plt.ylabel("Standardised Unemployment rate")
plt.title("SVM Classification of Economies in 2023")
plt.legend(handles = points.legend_elements()[0],
           labels = ["Emerging Market and Developing Economies",
                     "Advanced Economies"])

plt.show()

### Interpretation and Discussing the Output

Based on the Support Vector Machine results in Model 4 the model achieves an accuracy of 90.91 percent, with the same confusion matrix and classification report results as Model 3.

Whereas Figure 1 the graph above presents the SVM classification of economies in 2023. The horizontal x-axis measures standardized GDP per capita, whilst the vertical y-axis measures standardized unemployment rates. The red dots represent emerging market and developing economies, whilst the blue dots represent advanced economies within the training sample.

The decision boundary separating the two shaded regions distinguishes between the predicted classifications of the two groups. One can see that most advanced economies are concentrated towards higher values of GDP per capita. However, there is some overlap between the two groups particularly around the decision boundary.

Overall, despite the relatively high predictive accuracy across all four classification models, Models 1 and 2 perform marginally better than Models 3 and 4 within the testing sample.